# Regime-Switching Models: SPY Daily Returns
# financial-ts-models | Cole McLeod
# March 26, 2026

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
from arch import arch_model
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('../figures', exist_ok=True)

spy = yf.download('SPY', start='2019-01-01', end='2025-01-01')
spy.columns = spy.columns.get_level_values(0)
spy['Return'] = 100 * spy['Close'].pct_change()
spy = spy.dropna()
print(f'Observations: {len(spy)}')
print(f'Date range: {spy.index[0].date()} to {spy.index[-1].date()}')

## Notebook Plan

1. **Two-State Gaussian HMM** — Fit a hidden Markov model with two latent
   states to SPY daily returns. The model learns state-specific means and
   variances plus a transition matrix, giving us a probabilistic
   classification of each day as bull or bear.

2. **Regime Identification** — Label the two states as bull (high mean,
   low variance) and bear (low mean, high variance). Plot returns colored
   by regime and the filtered state probabilities over time.

3. **Transition Matrix Analysis** — Examine regime persistence and expected
   duration. Bull markets should be stickier than bear markets (higher
   self-transition probability).

4. **Regime-Conditional Volatility vs GARCH(1,1)** — Compare the HMM's
   within-regime volatility estimates to the GARCH(1,1) conditional
   volatility from notebook 02. The HMM captures discrete volatility
   shifts; GARCH captures smooth, autoregressive dynamics. Overlay both
   to see where they agree (steady regimes) and where they diverge
   (regime transitions).

## 1. Two-State Gaussian HMM

In [ ]:
returns = spy['Return'].values.reshape(-1, 1)

model = GaussianHMM(n_components=2, covariance_type='full', n_iter=200,
                     random_state=42)
model.fit(returns)

states = model.predict(returns)
spy['State'] = states

# Label states: bull = higher mean, bear = lower mean
means = model.means_.flatten()
if means[0] > means[1]:
    bull, bear = 0, 1
else:
    bull, bear = 1, 0

spy['Regime'] = spy['State'].map({bull: 'Bull', bear: 'Bear'})

print(f"State means:  Bull={means[bull]:.4f}%  Bear={means[bear]:.4f}%")
print(f"State stds:   Bull={np.sqrt(model.covars_[bull][0,0]):.4f}%  Bear={np.sqrt(model.covars_[bear][0,0]):.4f}%")
print(f"\nRegime counts:")
print(spy['Regime'].value_counts())

## 2. Regime Identification

In [ ]:
posteriors = model.predict_proba(returns)
bull_prob = posteriors[:, bull]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Returns colored by regime
bull_mask = spy['Regime'] == 'Bull'
axes[0].bar(spy.index[bull_mask], spy['Return'][bull_mask], width=1,
            color='steelblue', alpha=0.7, label='Bull')
axes[0].bar(spy.index[~bull_mask], spy['Return'][~bull_mask], width=1,
            color='firebrick', alpha=0.7, label='Bear')
axes[0].set_ylabel('Return (%)')
axes[0].set_title('SPY Daily Returns by HMM Regime')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Filtered bull-state probability
axes[1].fill_between(spy.index, bull_prob, alpha=0.4, color='steelblue')
axes[1].plot(spy.index, bull_prob, linewidth=0.8, color='steelblue')
axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_ylabel('P(Bull)')
axes[1].set_title('Filtered Bull-State Probability')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/hmm_regimes.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Transition Matrix Analysis

In [ ]:
transmat = model.transmat_
# Reorder so row/col 0 = Bull, 1 = Bear
ordered_transmat = np.array([
    [transmat[bull, bull], transmat[bull, bear]],
    [transmat[bear, bull], transmat[bear, bear]]
])

print("Transition Matrix (rows = from, cols = to):")
print(f"{'':>10} {'Bull':>10} {'Bear':>10}")
for i, label in enumerate(['Bull', 'Bear']):
    print(f"{label:>10} {ordered_transmat[i, 0]:>10.4f} {ordered_transmat[i, 1]:>10.4f}")

# Expected regime duration = 1 / (1 - p_ii)
bull_duration = 1 / (1 - ordered_transmat[0, 0])
bear_duration = 1 / (1 - ordered_transmat[1, 1])
print(f"\nExpected regime duration:")
print(f"  Bull: {bull_duration:.1f} days ({bull_duration/21:.1f} months)")
print(f"  Bear: {bear_duration:.1f} days ({bear_duration/21:.1f} months)")

# Stationary distribution
eigenvalues, eigenvectors = np.linalg.eig(ordered_transmat.T)
stationary = eigenvectors[:, np.argmin(np.abs(eigenvalues - 1))]
stationary = np.real(stationary / stationary.sum())
print(f"\nStationary distribution: Bull={stationary[0]:.3f}, Bear={stationary[1]:.3f}")

In [ ]:
# Visualize regime streaks
regime_changes = spy['Regime'].ne(spy['Regime'].shift()).cumsum()
streaks = spy.groupby(regime_changes).agg(
    regime=('Regime', 'first'),
    start=('Regime', lambda x: x.index[0]),
    duration=('Regime', 'count')
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for regime, color in [('Bull', 'steelblue'), ('Bear', 'firebrick')]:
    durations = streaks[streaks['regime'] == regime]['duration']
    axes[0].hist(durations, bins=20, alpha=0.7, color=color, label=regime)
axes[0].set_xlabel('Streak Length (days)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Regime Streak Lengths')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Transition matrix heatmap
im = axes[1].imshow(ordered_transmat, cmap='Blues', vmin=0, vmax=1)
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Bull', 'Bear'])
axes[1].set_yticklabels(['Bull', 'Bear'])
axes[1].set_xlabel('To')
axes[1].set_ylabel('From')
axes[1].set_title('Transition Probability Matrix')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{ordered_transmat[i,j]:.3f}',
                     ha='center', va='center', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.savefig('../figures/hmm_transitions.png', dpi=150, bbox_inches='tight')
plt.show()